This is a starter notebook for the project, you'll have to import the libraries you'll need, you can find a list of the ones available in this workspace in the requirements.txt file in this workspace. 

In [14]:
!pip3 install -r requirements.txt

  Using cached langchain-0.0.305-py3-none-any.whl (1.8 MB)
  Using cached pytest-8.3.2-py3-none-any.whl (341 kB)
  Using cached sentence_transformers-3.0.1-py3-none-any.whl (227 kB)
  Using cached transformers-4.44.2-py3-none-any.whl (9.5 MB)
  Using cached jupyter-1.0.0-py2.py3-none-any.whl (2.7 kB)
     |████████████████████████████████| 80 kB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 2.1 MB 4.1 MB/s eta 0:00:01
     |████████████████████████████████| 56 kB 15.3 MB/s eta 0:00:01
  Using cached numexpr-2.10.1-cp39-cp39-macosx_11_0_arm64.whl (130 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl (5.7 kB)
  Using cached aiohttp-3.10.5-cp39-cp39-macosx_11_0_arm64.whl (389 kB)
  Using cached ipykernel-6.29.5-py3-none-any.whl (117 kB)
     |████████████████████████████████| 123 kB 76.6 MB/s eta 0:00:01
  Using cached jupyter_console-6.6.3-py3-no

In [1]:
from langchain_community.llms import OpenAI
from langchain.vectorstores import Chroma
from langchain.prompts import FewShotPromptTemplate, PromptTemplate
from pydantic import BaseModel, Field, NonNegativeInt
from comet_ml import Experiment
from fastapi.encoders import jsonable_encoder
import torch
from diffusers import StableDiffusionPipeline
from langchain.output_parsers import PydanticOutputParser
from langchain_experimental.open_clip import OpenCLIPEmbeddings
from langchain.chains import RetrievalQA
from langchain.chains.question_answering import load_qa_chain
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import PIL
import os
import gc
from dotenv import load_dotenv
import pandas as pd
from typing import List

# Load environment variables
load_dotenv('my_config.env')

# API configuration
API_KEY = os.getenv('API_KEY')
openai_api_key = os.getenv("OPENAI_API_KEY")
COMET_API_KEY = os.getenv("COMET_API_KEY")


# Initialize the experiment
experiment = Experiment(
    api_key=COMET_API_KEY,
    project_name="real-estate-agent",
    workspace="polarbeargo",
    log_code=True,
)

/Users/hsin-wenchang/Documents/GitHub/GenAIND-Project-Personalized-Real-Estate-Agent/env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/polarbeargo/real-estate-agent/9697726df07d470da054321f56a1571e



- Define the prompt for generating synthetic real estate data (images and text)

In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [2]:
instruction = """
Generate five realistic real estate listings from a wide range of neighborhoods.
"""

template = \
"""
Here is the template of real estate listing:

Neighborhood: Mountain View
Price: $650,000
Bedrooms: 5
Bathrooms: 4
House Size: 3000 sqft
Description: Spacious family home with breathtaking views of the mountains and a large backyard for outdoor entertaining.
Neighborhood Description: Mountain View is known for its scenic landscape and outdoor activities, making it the ideal location for nature lovers and adventure seekers.
"""
llm = OpenAI(model_name="gpt-3.5-turbo-instruct", temperature=0.7, api_key=openai_api_key, max_tokens = 500)
image_model = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4")

# For Apple Silicon (M1/M2) replace mps to device if executing on other devices
image_model.to("mps")

image_dir = "generated_images"
os.makedirs(image_dir, exist_ok=True)

# Define the Listing data model
class Listing(BaseModel):
    neighborhood: str = Field(description="The neighborhood where the property is located.")
    price: NonNegativeInt = Field(description="The price of the property in USD.")
    bedrooms: NonNegativeInt = Field(description="The number of bedrooms in the property.")
    bathrooms: NonNegativeInt = Field(description="The number of bathrooms in the property.")
    house_size: NonNegativeInt = Field(description="The size of the property in square feet.")
    description: str = Field(description="A brief description of the property.")
    neighborhood_description: str = Field(description="A description of the neighborhood where the property is located.")

class Listings(BaseModel):
    listing: List[Listing] = Field(description="List of available real estate listings.")
    

def create_listing_prompt(listing: Listing) -> str:
    return f"""
    Neighborhood: {listing.neighborhood}
    Price: ${listing.price}
    Bedrooms: {listing.bedrooms}
    Bathrooms: {listing.bathrooms}
    House Size: {listing.house_size} sqft
    Description: {listing.description}
    Neighborhood Description: {listing.neighborhood_description}
    """

# Define few-shot examples
examples = [
    {
        "question": "Generate a listing for a 3-bedroom house in downtown.",
        "answer": Listing(
            neighborhood="Downtown",
            price=500000,
            bedrooms=3,
            bathrooms=2,
            house_size=1500,
            description="A beautiful 3-bedroom house located in the heart of downtown with modern amenities.",
            neighborhood_description="Downtown is vibrant and bustling, with plenty of restaurants, shops, and parks."
        )
    },
    {
        "question": "Create a listing for a luxury apartment in the suburbs.",
        "answer": Listing(
            neighborhood="Suburbia",
            price=750000,
            bedrooms=2,
            bathrooms=2,
            house_size=1200,
            description="A luxurious apartment featuring high-end finishes and spacious living areas.",
            neighborhood_description="Suburbia offers a peaceful environment with great schools and family-friendly parks."
        )
    }
]

parser = PydanticOutputParser(pydantic_object=Listings)

example_prompt = PromptTemplate(
    input_variables=["question", "answer"],
    template="{question}\n{answer}",
    partial_variables={"format_instructions": parser.get_format_instructions},
)

few_shot_prompt = FewShotPromptTemplate(
    examples=[{"question": ex["question"], "answer": create_listing_prompt(ex["answer"])} for ex in examples],
    example_prompt=example_prompt,
    suffix="Use these examples to generate a listing for the following question: {input}",
    input_variables=["input"],
    partial_variables={"format_instructions": parser.get_format_instructions},
)

full_prompt = few_shot_prompt.format(sample=template, input=instruction)
response = llm(full_prompt)
print(f"Raw Response: {response}")  # Debugging line to check the response format

/var/folders/f8/sxbz4hwx6js37sqlrvxpmq1m0000gn/T/ipykernel_20372/4082345002.py:17: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAI``.
  llm = OpenAI(model_name="gpt-3.5-turbo-instruct", temperature=0.7, api_key=openai_api_key, max_tokens = 500)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/Users/hsin-wenchang/Documents/GitHub/GenAIND-Project-Personalized-Real-Estate-Agent/env/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/var/folders/f8/sxbz4hwx6js37sqlrvxpmq1m0000gn/T/ipykernel_20372/4082345002.py:96: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm(full_prompt)


Raw Response: 
1. Neighborhood: Beachfront
Price: $1,200,000
Bedrooms: 4
Bathrooms: 3
House Size: 2000 sqft
Description: Enjoy breathtaking ocean views from this stunning 4-bedroom house located on the beachfront. With a spacious layout and modern amenities, this home is perfect for coastal living.
Neighborhood Description: Beachfront offers a relaxed and luxurious lifestyle with easy access to the beach and popular waterfront restaurants.

2. Neighborhood: Mountain View
Price: $900,000
Bedrooms: 5
Bathrooms: 4
House Size: 3000 sqft
Description: Nestled in the hills of Mountain View, this 5-bedroom house boasts stunning mountain views and a spacious backyard. With plenty of room for entertaining and a peaceful setting, this home is perfect for nature lovers.
Neighborhood Description: Mountain View offers a tranquil escape from the city with hiking trails, scenic views, and a tight-knit community.

3. Neighborhood: Historic District
Price: $600,000
Bedrooms: 3
Bathrooms: 2
House Size: 1

In [7]:
# Split the string into individual listings
listings = response.strip().split('\n\n')
print(listings)  # Debugging line to check the listings format
data = []

for listing in listings:

    # Remove the integer before 'Neighborhood'
    listing = listing.split('. ', 1)[1]

    # Split the listing into lines
    lines = listing.split('\n')
    listing_data = {}
    
    for line in lines:
        # Check if the line contains a colon
        if ': ' in line:
            # Split on the first colon
            key, value = line.split(': ', 1)  
            listing_data[key.strip()] = value.strip()
    
    data.append(listing_data)

df = pd.DataFrame(data)

df.rename(columns={
    'Neighborhood': 'neighborhood',
    'Price': 'price',
    'Bedrooms': 'bedrooms',
    'Bathrooms': 'bathrooms',
    'House Size': 'house_size',
    'Description': 'description',
    'Neighborhood Description': 'neighborhood_description'
}, inplace=True)

# Convert price to integer and house_size to integer (removing ' sqft')
df['price'] = df['price'].replace({'\$': '', ',': ''}, regex=True).astype(int)
df['house_size'] = df['house_size'].replace({' sqft': ''}, regex=True).fillna(0).astype(int)
df


['1. Neighborhood: Uptown\nPrice: $1,200,000\nBedrooms: 4\nBathrooms: 3\nHouse Size: 2500 sqft\nDescription: This stunning 4-bedroom home in Uptown boasts a modern design, high-end finishes, and a rooftop deck with city views.\nNeighborhood Description: Uptown is a trendy and upscale neighborhood with a vibrant nightlife and a variety of restaurants and shops.', '2. Neighborhood: Suburbia\nPrice: $600,000\nBedrooms: 3\nBathrooms: 2\nHouse Size: 1800 sqft\nDescription: This charming 3-bedroom house in Suburbia offers a peaceful escape from the city with a spacious backyard and updated kitchen.\nNeighborhood Description: Suburbia is a quiet and family-friendly neighborhood with excellent schools and local parks.', '3. Neighborhood: Waterfront\nPrice: $2,500,000\nBedrooms: 5\nBathrooms: 4\nHouse Size: 4000 sqft\nDescription: Live in luxury in this breathtaking 5-bedroom waterfront home featuring a private dock, pool, and expansive views of the ocean.\nNeighborhood Description: Waterfront 

,neighborhood,price,bedrooms,bathrooms,house_size,description,neighborhood_description
0,Uptown,1200000,4,3,2500,This stunning 4-bedroom home in Uptown boasts ...,Uptown is a trendy and upscale neighborhood wi...
1,Suburbia,600000,3,2,1800,This charming 3-bedroom house in Suburbia offe...,Suburbia is a quiet and family-friendly neighb...
2,Waterfront,2500000,5,4,4000,Live in luxury in this breathtaking 5-bedroom ...,Waterfront is an exclusive and upscale neighbo...
3,Historic District,900000,3,2,2200,Step back in time with this beautifully restor...,The Historic District is a charming and highly...
4,Arts District,800000,2,2,1500,This stylish 2-bedroom loft in the Arts Distri...,The Arts District is a vibrant and creative ne...


In [8]:
df.to_csv('generated_real_estate_data.csv', index_label = 'id')

In [9]:
# Batch generation of images based on the DataFrame
def generate_images(df, batch_size=2):
    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i + batch_size]
        for idx, row in batch.iterrows():
            prompt = f"A {row['bedrooms']}-bedroom house in {row['neighborhood']}. {row['neighborhood_description']}"
            image = image_model(prompt, num_inference_steps=50).images[0]
            image_path = os.path.join(image_dir, f"{row['neighborhood']}_{row['bedrooms']}_bedroom.png")
            image.save(image_path)
            print(f"Generated image saved at: {image_path}")

generate_images(df)


  0%|          | 0/50 [00:00<?, ?it/s]

Generated image saved at: generated_images/Uptown_4_bedroom.png


  0%|          | 0/50 [00:00<?, ?it/s]

Generated image saved at: generated_images/Suburbia_3_bedroom.png


  0%|          | 0/50 [00:00<?, ?it/s]

Generated image saved at: generated_images/Waterfront_5_bedroom.png


  0%|          | 0/50 [00:00<?, ?it/s]

Generated image saved at: generated_images/Historic District_3_bedroom.png


  0%|          | 0/50 [00:00<?, ?it/s]

Generated image saved at: generated_images/Arts District_2_bedroom.png


##### Multimodal Vector Database Store, Embeddings Transformation and Semantic Search

- Storing Listings Into a Vector Database

In [8]:
df = pd.read_csv('generated_real_estate_data.csv')
idx = [{'id':i} for i in range(len(df.index))]
image_paths = []
images = []
texts = []
text_template = """
Neighborhood: {}
Price: {}
Bedrooms: {}
Bathrooms: {}
House Size: {}

Description: {}
Neighborhood Description: {}
"""

for i, row in df.iterrows():
    image_path = os.path.join(image_dir, f"{row['neighborhood']}_{row['bedrooms']}_bedroom.png")
    image_paths.append(image_path)
    images.append(PIL.Image.open(image_path))
    texts.append(text_template.format(row['neighborhood'], row['price'], row['bedrooms'], row['bathrooms'], row['house_size'], row['description'], row['neighborhood_description']))

In [9]:
embeddings = OpenCLIPEmbeddings()
clip_db = Chroma(collection_name="real_estate_listings", embedding_function=embeddings)

clip_db.add_texts(
    texts=texts,
    metadatas=idx
)

clip_db.add_images(
    uris=image_paths,
    metadatas=idx
)

/var/folders/f8/sxbz4hwx6js37sqlrvxpmq1m0000gn/T/ipykernel_20372/232635149.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  clip_db = Chroma(collection_name="real_estate_listings", embedding_function=embeddings)


['1ee304bf-802d-40b3-aa4e-717b0ba4c8ce',
 '4d606f37-4bae-4777-b478-bfc52aafb7ce',
 '5114997c-610c-482e-b110-0a3365eb2ea4',
 'b74e48c6-dce9-49d4-ba57-9e2cecec31b6',
 '94684ab3-072e-4464-94c2-c97a5205ac5c']

In [24]:
res = clip_db.similarity_search("", k=3)
res

[Document(metadata={'id': 1}, page_content='\nNeighborhood: Suburbia\nPrice: 600000\nBedrooms: 3\nBathrooms: 2\nHouse Size: 1800\n\nDescription: This charming 3-bedroom house in Suburbia offers a peaceful escape from the city with a spacious backyard and updated kitchen.\nNeighborhood Description: Suburbia is a quiet and family-friendly neighborhood with excellent schools and local parks.\n'),
 Document(metadata={'id': 2}, page_content='...'),
 Document(metadata={'id': 0}, page_content='...')]

In [19]:
questions = [   
                "How big do you want your house to be?" 
                "What are 3 most important things for you in choosing this property?", 
                "Which amenities would you like?", 
                "Which transportation options are important to you?",
                "How urban do you want your neighborhood to be?",   
            ]
answers = [
    "A comfortable three-bedroom house with a spacious kitchen and a cozy living room.",
    "A quiet neighborhood, good local schools, and convenient shopping options.",
    "A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system.",
    "Easy access to a reliable bus line, proximity to a major highway, and bike-friendly roads.",
    "A balance between suburban tranquility and access to urban amenities like restaurants and theaters."
]

- Implementing a Retrieval-Augmented Generation (RAG) system using LangChain to perform a semantic similarity search to find vectors that are semantically similar to our query and send the text associated with the vectors to LLM for summarization.

In [20]:
query = f"""
{questions[0]} {answers[0]}
{questions[1]} {answers[1]}
{questions[2]} {answers[2]}
{questions[3]} {answers[3]}
"""

use_chain_helper = False

if use_chain_helper:
    rag = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=clip_db.as_retriever())
    print(rag.run(query))
else:
    similar_docs = clip_db.similarity_search(query, k=3)
    prompt = PromptTemplate(
        template="{query}\nContext: {context}",
        input_variables=["query", "context"],
    )
    chain = load_qa_chain(llm, prompt = prompt, chain_type="stuff")
    print(chain.run(input_documents=similar_docs, query=query))


- Performance Evaluation Metrics
    - With generate Real Estate Listings.
    - Define Reference Listings.
    - Calculate Evaluation Metrics.

In [23]:

def evaluate_generated_listings(generated: str, reference: str) -> dict:
    
    # Calculate BLEU score
    smoothing_function = SmoothingFunction()
    bleu_score = sentence_bleu([reference.split()], generated.split(), smoothing_function=smoothing_function.method1)
    
    # Calculate ROUGE score
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, generated)
    
    return {
        "bleu": bleu_score,
        "rouge1": scores['rouge1'].fmeasure,
        "rouge2": scores['rouge2'].fmeasure,
        "rougeL": scores['rougeL'].fmeasure
    }

reference_listings = [
    "Neighborhood: Beachfront\nPrice: $1,200,000\nBedrooms: 4\nBathrooms: 3\nHouse Size: 3000 sqft\nDescription: A stunning beachfront property with ocean views.\nNeighborhood Description: Beachfront is known for its luxurious homes and vibrant community.",
    "Neighborhood: City Center\nPrice: $800,000\nBedrooms: 2\nBathrooms: 1\nHouse Size: 900 sqft\nDescription: A modern apartment located in the heart of the city.\nNeighborhood Description: City Center is bustling with activity and offers a variety of amenities."
    "Neighborhood: Mountain View\nPrice: $650,000\nBedrooms: 5\nBathrooms: 4\nHouse Size: 3000 sqft\nDescription: Spacious family home with breathtaking views of the mountains and a large backyard for outdoor entertaining.\nNeighborhood Description: Mountain View is known for its scenic landscape and outdoor activities, making it the ideal location for nature lovers and adventure seekers."
    "Neighborhood: Suburbia\nPrice: $750,000\nBedrooms: 2\nBathrooms: 2\nHouse Size: 1200 sqft\nDescription: A luxurious apartment featuring high-end finishes and spacious living areas.\nNeighborhood Description: Suburbia offers a peaceful environment with great schools and family-friendly parks."
    "Neighborhood: Downtown\nPrice: $500,000\nBedrooms: 3\nBathrooms: 2\nHouse Size: 1500 sqft\nDescription: A beautiful 3-bedroom house located in the heart of downtown with modern amenities.\nNeighborhood Description: Downtown is vibrant and bustling, with plenty of restaurants, shops, and parks."
]

# Evaluate the generated listings
for generated, reference in zip(response, reference_listings):
    metrics = evaluate_generated_listings(generated, reference)
    print(f"Generated Listing: {generated}")
    print(f"Evaluation Metrics: {metrics}")

Generated Listing: 

Evaluation Metrics: {'bleu': 0, 'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0}
Generated Listing: 1
Evaluation Metrics: {'bleu': 1.0609745534868908e-72, 'rouge1': 0.011363636363636362, 'rouge2': 0.0, 'rougeL': 0.011363636363636362}


- Integrate kubeflow pipelines

In [ ]:
%%writefile Generator.py

In [ ]:
%%writefile CLIP.py

In [ ]:
%%writefile HomeMatch.py


- Use Gradio to create an interactive interface where we can input data related to home preferences, and the model can predict the best match for them.

In [18]:
import gradio as gr 

In [44]:
def retrieve_data(query):
    # Retrieve similar texts and images based on the query
    similar_texts = clip_db.similarity_search(query, k=5)
    similar_images = clip_db.similarity_search_images(query, k=5)

    output_texts = "\n".join(similar_texts)
    output_images = [image for image in similar_images]

    return output_texts, output_images

def generate_app():
    with gr.Blocks() as demo:
        gr.Markdown("""
        # Your top 5 Real Estate Listings
        1. Fill out your query based on customer preferences.
        2. Click on the "Search" button to fetch the top 5 real estate listings based on your input preferences.
        3. Review the listings displayed below. Each listing includes an image, price, location, and a brief description.
        4. If you want to refine your search, adjust your preferences and click "Search" again.
        """)

        query_input = gr.Textbox(label="Enter your query about real estate:")
        search_button = gr.Button("Search")
    
        output_text = gr.Textbox(label="Retrieved Listings", interactive=False)
        output_image = gr.Gallery(label="Retrieved Images", show_label=True)

        search_button.click(fn=retrieve_data, inputs=query_input, outputs=[output_text, output_image])

    demo.launch()
    return demo

In [45]:
if __name__ == "__main__":
    interface = generate_app()

Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.


In [38]:
interface.close()

Closing server running on port: 7862


In [25]:

# Delete the vector store variable if it's no longer needed
del clip_db 

experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : statistical_abbey_9315
COMET INFO:     url                   : https://www.comet.com/polarbeargo/real-estate-agent/9697726df07d470da054321f56a1571e
COMET INFO:   Uploads:
COMET INFO:     environment details      : 1
COMET INFO:     filename                 : 1
COMET INFO:     git metadata             : 1
COMET INFO:     git-patch (uncompressed) : 1 (1.57 KB)
COMET INFO:     installed packages       : 1
COMET INFO:     notebook                 : 1
COMET INFO:     source_code              : 1
COMET INFO: 
COMET INFO: Please wait for assets to finish uploading (timeout is 10800 seconds)
COMET INFO: Still uploading 1 file(s), remaining 2.05 MB/2.97 MB
CO